In [1]:
!pip install sentence-transformers faiss-cpu google-generativeai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import zipfile

# Download dataset
!wget -q https://snap.stanford.edu/data/facebook_large.zip

extract_path = "/kaggle/working/facebook_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile("facebook_large.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset ready at:", extract_path)
print(os.listdir(extract_path))

✅ Dataset ready at: /kaggle/working/facebook_data
['facebook_large']


In [3]:
DATA_PATH = "/kaggle/working/facebook_data/facebook_large"

features_path = os.path.join(DATA_PATH, "musae_facebook_features.json")
target_path = os.path.join(DATA_PATH, "musae_facebook_target.csv")

In [4]:
import csv

labels_dict = {}

with open(target_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        labels_dict[int(row["id"])] = row["page_type"]

nodes = sorted(labels_dict.keys())

print("Total nodes:", len(nodes))

Total nodes: 22470


In [5]:
documents = []

for node in nodes:
    label = labels_dict[node]
    
    doc = f"""
    Node ID: {node}
    This Facebook page belongs to category: {label}.
    This page is part of a social network graph dataset.
    """
    
    documents.append(doc)

print(documents[:2])

['\n    Node ID: 0\n    This Facebook page belongs to category: tvshow.\n    This page is part of a social network graph dataset.\n    ', '\n    Node ID: 1\n    This Facebook page belongs to category: government.\n    This page is part of a social network graph dataset.\n    ']


In [6]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = encoder.encode(documents, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/703 [00:00<?, ?it/s]

Embedding shape: (22470, 384)


In [7]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready")

✅ FAISS index ready


In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready")

✅ FAISS index ready


In [ ]:
import google.generativeai as genai

genai.configure(api_key="your-gemini-api-key")

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini ready")

Gemini ready


In [28]:
def retrieve(query, top_k=5):
    q_emb = encoder.encode([query])
    
    distances, indices = index.search(np.array(q_emb), top_k)
    
    return [documents[i] for i in indices[0]]

In [29]:
def rag_query(query, top_k=5):
    context_docs = retrieve(query, top_k)
    
    context = "\n".join(context_docs)
    
    prompt = f"""
    You are a graph intelligence assistant.

    Context from Facebook graph dataset:
    {context}

    Question:
    {query}

    Answer clearly using the context.
    """
    
    response = model.generate_content(prompt)
    
    print("🔍 Query:", query)
    print("\n📄 Retrieved Context:\n")
    for doc in context_docs:
        print("-", doc.strip())
    
    print("\n🤖 Gemini Answer:\n")
    print(response.text)

In [30]:
rag_query("What type of facebook pages exist in this dataset?")

🔍 Query: What type of facebook pages exist in this dataset?

📄 Retrieved Context:

- Node ID: 16102
    This Facebook page belongs to category: government.
    This page is part of a social network graph dataset.
- Node ID: 1919
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 19160
    This Facebook page belongs to category: government.
    This page is part of a social network graph dataset.
- Node ID: 19343
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 20112
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.

🤖 Gemini Answer:

Based on the provided context, the types of Facebook pages that exist in this dataset are:

*   **government**
*   **tvshow**


In [31]:
rag_query("Tell me about categories present in the network")

🔍 Query: Tell me about categories present in the network

📄 Retrieved Context:

- Node ID: 911
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 8022
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 2003
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 1864
    This Facebook page belongs to category: tvshow.
    This page is part of a social network graph dataset.
- Node ID: 8029
    This Facebook page belongs to category: company.
    This page is part of a social network graph dataset.

🤖 Gemini Answer:

Based on the provided context, the categories present in the network are:

*   **tvshow**
*   **company**


In [32]:
rag_query("Which nodes belong to politician category?")

🔍 Query: Which nodes belong to politician category?

📄 Retrieved Context:

- Node ID: 978
    This Facebook page belongs to category: politician.
    This page is part of a social network graph dataset.
- Node ID: 20049
    This Facebook page belongs to category: politician.
    This page is part of a social network graph dataset.
- Node ID: 20069
    This Facebook page belongs to category: politician.
    This page is part of a social network graph dataset.
- Node ID: 20066
    This Facebook page belongs to category: politician.
    This page is part of a social network graph dataset.
- Node ID: 20027
    This Facebook page belongs to category: politician.
    This page is part of a social network graph dataset.

🤖 Gemini Answer:

Based on the context, the following nodes belong to the politician category:

*   978
*   20049
*   20069
*   20066
*   20027
